# Widgets Configuration

In [0]:


schema_choices = ["raw", "bronze", "silver", "gold", "security"]

dbutils.widgets.text("project_catalog", "vstone_catalog", "1. Target Catalog Name")
dbutils.widgets.text("raw_schema", "raw", "2. Raw Schema Name")
dbutils.widgets.multiselect("schemas", "bronze", schema_choices, "3. Target Schemas")

# Fetch values into variables
CATALOG = dbutils.widgets.get("project_catalog")
RAW_SCHEMA = dbutils.widgets.get("raw_schema")
SELECTED_SCHEMAS = dbutils.widgets.get("schemas").split(",")

#Catalog & Infrastructure Logic

In [0]:

print(f"--- INITIALIZING INFRASTRUCTURE FOR CATALOG: {CATALOG} ---")

# 1. CATALOG INITIALIZATION
try:
    # Attempting to create (will skip if exists)
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG} COMMENT 'VStone Russian Car Market Project'")
    print(f"Catalog Verified/Created: {CATALOG}")
except Exception as e:
    if "PERMISSION_DENIED" in str(e):
        print(f" Using Existing Catalog: {CATALOG} (No CREATE permissions on Metastore)")
    else:
        raise e

# Mandatory: Point Spark to the correct catalog for subsequent schema/volume operations
spark.sql(f"USE CATALOG {CATALOG}")

# 2. DYNAMIC SCHEMA CREATION
# Merging raw_schema with selected schemas from widgets to ensure full coverage
target_schemas = list(set([RAW_SCHEMA] + SELECTED_SCHEMAS))

for schema_name in target_schemas:
    spark.sql(f"""
        CREATE SCHEMA IF NOT EXISTS {schema_name} 
        COMMENT 'VStone {schema_name.upper()} layer'
    """)
    print(f"   Schema Ready: {CATALOG}.{schema_name}")

# 3. VOLUME CREATION
# Creating standard landing and chunks volumes inside the RAW schema
volumes = ["landing", "chunks"]

for vol_name in volumes:
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {RAW_SCHEMA}.{vol_name}
        COMMENT 'UC Volume for {vol_name} data'
    """)
    print(f"   Volume Ready: /Volumes/{CATALOG}/{RAW_SCHEMA}/{vol_name}")

# FINAL SUMMARY REPORT
print(f"""
{'-'*80}
 SETUP COMPLETE: vstone_project is ready for processing
{'-'*80}
Target Catalog : {CATALOG}
Raw/Landing    : /Volumes/{CATALOG}/{RAW_SCHEMA}/landing
Chunks/Output  : /Volumes/{CATALOG}/{RAW_SCHEMA}/chunks
Layers Config  : {', '.join(SELECTED_SCHEMAS)}
{'-'*80}
""")